In [ ]:
from typing import TypedDict

class Person(TypedDict):
    name: str
    age: int

new_person: Person = {'name': 'Koyel', 'age': 26}
print(new_person) # no validation, only indication


In [ ]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict

load_dotenv()

model = ChatOpenAI

# schema (simple example)
class Review(TypedDict):
    summary: str
    sentiment: str

structured_model = model.with_structured_output(Review)

result = structured_model.invoke("""The hardware is great, but the software feels bloated. 
                                  There are too many pre-installed apps that i can't remove. Also, the UI
                                  looks outdated compared to other brands. Having for a software update to fix this."""
)

In [ ]:
print(type(result))

In [ ]:
print(result.keys())

In [ ]:
print(result['summary'])

In [ ]:
print(result['sentiment'])

In [ ]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Optional

load_dotenv()

model = ChatOpenAI

# schema (Annotated Example)
class Review(TypedDict):
    key_themes: Annotated[list[str], 'Write down all the key themes discussed in the review in a list']
    summary: Annotated[str, 'A brief summary of the review']
    # sentiment: Annotated[str, 'Return sentiment of the review either negative or positive']
    sentiment: Annotated[literal['pos', 'neg'], 'Return sentiment of the review either negative or positive']
    pros: Annotated[Optional[list[str]], 'Write down all the pros inside a list']
    cons: Annotated[Optional[list[str]], 'Write down all the cons inside a list']
    name: Annotated[Optional[str], 'Write the name of the reviewer']
    

structured_model = model.with_structured_output(Review)

result = structured_model.invoke("""The hardware is great, but the software feels bloated. 
                                  There are too many pre-installed apps that i can't remove. Also, the UI
                                  looks outdated compared to other brands. Having for a software update to fix this."""
)

In [ ]:
# run the above query without specifying `cons` run and test
# if it's still creating, then mention in prompt engineering, `don't generate cons unless explicitly mentioned in prompt`
# validation is not there in TypedDict, datatype can be changed in output, also if suppose reviewer not mentioned, then also it can return if mentioned in Schema usimh TypedDict

In [ ]:
# Pydantic has additional propety of validation along with proprtties of TypedDict
from pydantic import BaseModel
from typing import Optional

class Student(BaseModel):
    name: str
    age: Optional[int] = None

# new_student = {'name': 'Koyel'}
new_student = {'name': '32'} # this will not throw error as it's string
# new_student = {'age': 32} # throw error for datatype mismatch 

student = Student(**new_student)

print(student)

In [ ]:
print(student.name)

In [ ]:
print(type(student.name))

In [ ]:
new_student = {'age': '32'} 
# this will not throw error though age is going in string as Pydantic supports coerce (automatic type change if data is correct only dtype is showing different)

student = Student(**new_student)

print(student)

In [ ]:
from pydantic import BaseModel, EmailStr
from typing import Optional

class Student(BaseModel):
    name: str
    age: Optional[int] = None
    email: EmailStr

# new_student = {'name': 'Koyel'}
# new_student = {'name': 32}
new_student = {'age': 32, 'email': 'abc'}

student = Student(**new_student)

print(student)

In [ ]:
from pydantic import BaseModel, EmailStr, Field
from typing import Optional

class Student(BaseModel):
    name: str
    age: Optional[int] = None
    email: EmailStr
    cgpa: float = Field(gt=0, lt=10, default = 6, description='A decimal value representing the cgpa of the student')

# new_student = {'name': 'Koyel'}
# new_student = {'name': 32}
new_student = {'age': 32, 'email': 'abc', 'cgpa': 5}

student = Student(**new_student) # pydantic object, if we required dictionary hve to convert using model_json_demp, otherwise ahve to use like python object

student_dict = dict(student)
print(student_dict)
print(student_dict) # pydantic object

In [ ]:
student_json = student.model_dump_json() # load pydantic object as json file

In [ ]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Optional
from pydantic import BaseModel, EmailStr, Field


load_dotenv()

model = ChatOpenAI()

# schema
class Review(BaseModel):
    key_themes: list[str]=Field(description='Write down all the key themes discussed in the review in a list')
    summary: str=Field(description='A brief summary of the review')
    # sentiment: Annotated[str, 'Return sentiment of the review either negative or positive']
    sentiment: Literal['pos', 'neg'] = Field(description='Return sentiment of the review either negative, positive or neutral')
    pros: Optional[list[str]] =  Field(description='Write down all the pros inside a list')
    cons: Optional[list[str]] =  Field(description='Write down all the cons inside a list')
    name: Optiona[str] = Field(description='Write the name of the reviewer')
    

structured_model = model.with_structured_output(Review)

result = structured_model.invoke("""The hardware is great, but the software feels bloated. 
                                  There are too many pre-installed apps that i can't remove. Also, the UI
                                  looks outdated compared to other brands. Having for a software update to fix this."""
)

In [ ]:
print(result)

In [ ]:
print(result.name)

In [ ]:
# json_schema.json
# when full project is not in same language, i.e. backend is written in java
# frontend is written in Python, in that case needed json_schema bcz json is common structure for all languages
# load below json in a .json file
{
    "title": "student",
    "description": "schema about students",
    "type": "object",
    "properties": {
        "name": "string",
        "age": "integer"
    },
    "requied":['name']
}

# description is optional but can be useful in our case of review

In [ ]:
# create a seperate .py file i.e. with_structured_output_json.py
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Optional
from pydantic import BaseModel, EmailStr, Field


load_dotenv()

model = ChatOpenAI()

# schema (JSON)
json_schema = {
    "title": "Review",
    "type": "object",
    "properties": {
        "key_themes": {
            "type": "array",
            "items": {
            "type": "string"
        },
        "description": "Write down all the key themes discussed in the review in a list"
        },
        "summary": {
            "type": "string",
            "description": "A brief summary of the review"
        },
        "sentiment": {
            "type": "string",
            "enum": ["pos", "neg"],
            "description": "Return sentiment of the review either negative, positive or neutral"
        },
        "pros": {
            "type": ["array", "null"],
            "items": {
                "type": "string"
            },
            "description": "Write down all the pros inside a list"
        },
        "cons": {
            "type": ["array", "null"],
            "items": {
                "type": "string"
            },
            "description": "Write down all the cons inside a list"
        },
        "name": {
            "type": ["string", "null"],
            "description": "Write the name of the reviewer"
        },
        "required": ["key_themes", "summary", "sentiment"]
        }
    }




structured_model = model.with_structured_output(json_schema)

result = structured_model.invoke("""The hardware is great, but the software feels bloated. 
                                  There are too many pre-installed apps that i can't remove. Also, the UI
                                  looks outdated compared to other brands. Having for a software update to fix this."""
)

print(result)